# Impact of Class Imbalance on Synthetic Data Generation

This notebook aggregates results across all datasets.

## Setup and Data Loading

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
from pathlib import Path
import re
from scipy.stats import linregress
from math import pi
warnings.filterwarnings('ignore')
import sys
import os
from pathlib import Path
root = next(p for p in Path().resolve().parents if (p / 'config').exists())
sys.path.append(str(root))
os.chdir(root)
from config.config import get_config
config = get_config()

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 370
GENERATORS = config.models.generators
IMBALANCE_RATIOS = config.experiment.imbalance_ratios

GENERATOR_STYLES = {}
for gen in GENERATORS:
    style = config.visualization.generator_styles.__dict__[gen]
    GENERATOR_STYLES[gen] = {
        'marker': style.marker,
        'linestyle': style.linestyle,
        'color': style.color,
        'label': gen.upper()
    }

RUN_IDS = {
    'auto_mpg': 'run_20260529_111920',
    'credit': 'run_20260529_182128',
    'mammographic_mass': 'run_20260427_150649',
    'monks': 'run_20260530_090834',
    'phishing': 'run_20260524_132915',
    'vote': 'run_20260529_131240',
}

ModuleNotFoundError: No module named 'config.config'

### Load All Dataset Results

In [ ]:
result_files = {}
for dataset, run in RUN_IDS.items():
    result_files[dataset] = f'results/{dataset}/tables/{run}/detailed_experiment_results.csv'

dfs = []
for dataset_name, filepath in result_files.items():
    df = pd.read_csv(filepath)
    df['dataset'] = dataset_name
    dfs.append(df)

all_results = pd.concat(dfs, ignore_index=True)

# majority and minority
majority_dfs = []
minority_dfs = []
for dataset_name, filepath in result_files.items():
    base = Path(filepath).parent
    for dfs_list, filename in [
        (majority_dfs, "detailed_experiment_results_majority.csv"),
        (minority_dfs, "detailed_experiment_results_minority.csv")
    ]:
        path = base / filename
        if path.exists():
            df = pd.read_csv(path)
            dfs_list.append(df)
        else:
            print(f"Warning: {filename} not found for {dataset_name}")

majority_all_results = pd.concat(majority_dfs, ignore_index=True)
minority_all_results = pd.concat(minority_dfs, ignore_index=True)

print(f"Total records: {len(all_results)}")
print(f"Datasets: {all_results['dataset'].unique()}")
print(f"Models: {all_results['model'].unique()}")
print(f"\nRecords per dataset:")
print(all_results.groupby('dataset').size())

print(all_results.columns.tolist())

In [ ]:
all_results.head()

In [ ]:
all_results.info()

### Dataset Characteristics Summary

In [ ]:
import problexity as px
from sklearn.preprocessing import LabelEncoder

raw_target_features = {
    'credit': '+',
    'vote': 'target',
    'mammographic_mass': 'Severity',
    'monks': "'class'",
    'phishing': 'Result',
    'auto_mpg': 'binaryClass'
}

raw_data_info = {}
raw_files = {
    'credit': 'data/raw/credit.csv',
    'vote': 'data/raw/vote.csv',
    'mammographic_mass': 'data/raw/mammographic_mass.csv',
    'monks': 'data/raw/monks-problems-1.csv',
    'phishing': 'data/raw/phising.csv',
    'auto_mpg': 'data/raw/auto_mpg.csv'
}

for name, path in raw_files.items():
    try:
        df = pd.read_csv(path).dropna()
        target = raw_target_features[name]
        X = df.drop(columns=[target])
        X = X.apply(lambda col: LabelEncoder().fit_transform(col) if col.dtype == 'object' else col)
        X = X.astype(float)
        y = df[target]
        y_encoded = LabelEncoder().fit_transform(df[target])
        
        cc = px.ComplexityCalculator()
        cc.fit(X, y_encoded)
        report = cc.report()
        
        raw_data_info[name] = {
            'n_samples': len(df),
            'n_features': len(df.columns) - 1,  # Exclude target
            'columns': list(df.columns),
            'f1': round(report['complexities']['f1'], 3),
            'px_ir': report['complexities']['c2'],
            'imbalance_ratio': round(y.value_counts().max() / y.value_counts().min(), 2)
        }
    except Exception as e:
        print(f"Failed for {name}: {e}")
        raw_data_info[name] = {'n_samples': 'N/A', 'n_features': 'N/A', 'f1': 'N/A', 'px_ir': 'N/A', 'imbalance_ratio': 'N/A'}

dataset_chars = pd.DataFrame(raw_data_info).T
dataset_chars.index.name = 'Dataset'
dataset_chars = dataset_chars[['n_samples', 'n_features', 'f1', 'px_ir', 'imbalance_ratio']]
dataset_chars.columns = ['Samples', 'Features', 'F1 (Maximum Fishers Discriminant Ratio)', 'px_IR', 'IR']

def size_category(n):
    if n == 'N/A':
        return 'Unknown'
    elif n < 2000:
        return 'Small'
    elif n < 10000:
        return 'Medium'
    else:
        return 'Large'

dataset_chars['Size Category'] = dataset_chars['Samples'].apply(size_category)

print("Dataset Characteristics:")
dataset_chars

### Separate Synthetic and Baseline Results

In [ ]:
baseline_results = all_results[all_results['model'] == 'baseline'].copy()
synthetic_results = all_results[all_results['model'] != 'baseline'].copy()

# Filter to only imbalanced condition for main analysis (control used separately)
imbalanced_results = synthetic_results[synthetic_results['dataset_type'] == 'imbalanced'].copy()
control_results = synthetic_results[synthetic_results['dataset_type'] == 'control'].copy()

print(f"Baseline records: {len(baseline_results)}")
print(f"Synthetic records: {len(synthetic_results)}")
print(f"  - Imbalanced condition: {len(imbalanced_results)}")
print(f"  - Control condition: {len(control_results)}")

---

# Part 1: Imbalance-Quality Relationship

## 1.1 Degradation Curves

In [ ]:
def compute_aggregated_metrics(df, groupby_cols=['ir_numeric', 'model']):
    """
    Compute mean and std of metrics across datasets and repetitions.
    """
    # 1. Downstream Utility Metrics (TSTR)
    utility_metrics = [
        'f1_minority', 
        'roc_auc', 
        'precision_minority', 
        'recall_minority'
    ]
    
    # 2. Data Quality Metrics (pyMDMA)
    quality_metrics = [
        'improved_precision',           # Fidelity
        'density',                      # Fidelity
        'improved_recall',              # Diversity
        'coverage',                     # Diversity
        'authenticity',                 # Privacy
        'statisticalSimilarity',        # Fidelity
        'statisticalDivergence',        # Fidelity
        'coherence'                     # Fidelity
    ]
    
    all_metrics = utility_metrics + quality_metrics
    
    available_metrics = [m for m in all_metrics if m in df.columns]
    
    agg_funcs = {}
    for m in available_metrics:
        agg_funcs[f'{m}_mean'] = (m, 'mean')
        agg_funcs[f'{m}_std'] = (m, 'std')
        agg_funcs[f'{m}_count'] = (m, 'count')

    return df.groupby(groupby_cols).agg(**agg_funcs).reset_index()

master_agg = compute_aggregated_metrics(imbalanced_results)

# majority and minority
majority_imbalanced = majority_all_results[majority_all_results['model'].isin(GENERATORS)].copy()
minority_imbalanced = minority_all_results[minority_all_results['model'].isin(GENERATORS)].copy()

majority_agg = compute_aggregated_metrics(majority_imbalanced, groupby_cols=['ir_numeric', 'model'])
minority_agg = compute_aggregated_metrics(minority_imbalanced, groupby_cols=['ir_numeric', 'model'])

print("Aggregated Columns:", master_agg.columns.tolist())

In [ ]:
def plot_degradation_grid(agg_df, metrics_config, title_main, control_df=None, n_cols=2):

    n_metrics = len(metrics_config)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
    axes = axes.flatten()

    for idx, (metric, display_name, ylim) in enumerate(metrics_config):
        ax = axes[idx]
        mean_col = f'{metric}_mean'
        std_col = f'{metric}_std'
        
        # Skip if metric wasn't computed
        if mean_col not in agg_df.columns:
            ax.text(0.5, 0.5, f"{metric} not found", ha='center')
            continue

        if control_df is not None and metric in control_df.columns:
            ctrl = control_df.groupby('ir_numeric')[metric].agg(['mean', 'std']).reset_index()
            if not ctrl.empty:
                ax.plot(ctrl['ir_numeric'], ctrl['mean'],
                        color='dimgray', linestyle='--', linewidth=2.5,
                        marker=None , markersize=7,
                        label='Control (size-matched, balanced)',
                        zorder=1, alpha=0.9)
                ax.fill_between(ctrl['ir_numeric'],
                                ctrl['mean'] - ctrl['std'],
                                ctrl['mean'] + ctrl['std'],
                                alpha=0.08, color='dimgray')

        for gen in GENERATORS:
            gen_data = agg_df[agg_df['model'] == gen].sort_values('ir_numeric')
            if gen_data.empty: continue
            
            style = GENERATOR_STYLES[gen]

            ax.plot(gen_data['ir_numeric'], gen_data[mean_col],
                   marker=style['marker'], linestyle=style['linestyle'],
                   color=style['color'], label=style['label'],
                   linewidth=2, markersize=8)

            # Error bands
            ax.fill_between(gen_data['ir_numeric'],
                           gen_data[mean_col] - gen_data[std_col],
                           gen_data[mean_col] + gen_data[std_col],
                           alpha=0.15, color=style['color'])

        ax.set_xlabel('Imbalance Ratio')
        ax.set_ylabel(display_name)
        ax.set_title(f'{display_name} vs Imbalance')
        ax.set_xscale('log')
        ax.set_xticks(IMBALANCE_RATIOS)
        ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS])
        if ylim:
            ax.set_ylim(ylim)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)

    # Hide empty subplots
    for i in range(len(metrics_config), len(axes)):
        axes[i].set_visible(False)

    plt.suptitle(title_main, y=1.02, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Downstream Utility 
utility_metrics_config = [
    ('f1_minority', 'F1-Score (Minority)', (0, 1)),
    ('roc_auc', 'ROC-AUC', (0.5, 1)),
    ('precision_minority', 'Precision (Minority)', (0, 1)),
    ('recall_minority', 'Recall (Minority)', (0, 1))
]

# create path
Path('results/aggregate_results/figures').mkdir(parents=True, exist_ok=True)

fig_util = plot_degradation_grid(master_agg, utility_metrics_config, 
                                 '', control_df=control_results)
plt.savefig('results/aggregate_results/figures/fig1a_utility_degradation.png', bbox_inches='tight')
plt.show()


In [ ]:
# Data Quality 
quality_metrics_config = [
    # Fidelity
    ('improved_precision', 'Fidelity: Improved Precision', (0, 1)),
    ('density', 'Fidelity: Density', (0, 2)), # Density can go >1
    # Diversity
    ('improved_recall', 'Diversity: Improved Recall', (0, 1)),
    ('coverage', 'Diversity: Coverage', (0, 1)),
    # Privacy
    ('authenticity', 'Privacy: Authenticity', (0, 1))
]

In [ ]:
quality_trio_config = [
    ('density', 'Fidelity: Density', (0, 2)),
    ('coverage', 'Diversity: Coverage', (0, 1)),
    ('authenticity', 'Privacy: Authenticity', (0, 1))
]

fig_trio = plot_degradation_grid(master_agg, quality_trio_config,
                                 '',
                                 n_cols=3)
fig_trio.savefig('results/aggregate_results/figures/fig1c_quality_trio.png', bbox_inches='tight')
plt.show()

In [ ]:
for metric, display_name, ylim in quality_metrics_config:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{display_name}: Majority vs Minority Class',
                 fontsize=14, fontweight='bold')

    for ax, agg_df, class_label in [
        (axes[0], majority_agg, 'Majority Class'),
        (axes[1], minority_agg, 'Minority Class')
    ]:
        mean_col = f'{metric}_mean'
        std_col = f'{metric}_std'

        if mean_col not in agg_df.columns:
            ax.text(0.5, 0.5, f'{metric} not found', ha='center')
            continue

        for gen in GENERATORS:
            gen_data = agg_df[agg_df['model'] == gen].sort_values('ir_numeric')
            if gen_data.empty:
                continue
            style = GENERATOR_STYLES[gen]
            ax.plot(gen_data['ir_numeric'], gen_data[mean_col],
                   marker=style['marker'], linestyle=style['linestyle'],
                   color=style['color'], label=style['label'],
                   linewidth=2, markersize=8)
            ax.fill_between(gen_data['ir_numeric'],
                           gen_data[mean_col] - gen_data[std_col],
                           gen_data[mean_col] + gen_data[std_col],
                           alpha=0.15, color=style['color'])

        ax.set_xlabel('Imbalance Ratio')
        ax.set_ylabel(display_name)
        ax.set_title(class_label)
        ax.set_xscale('log')
        ax.set_xticks(IMBALANCE_RATIOS)
        ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS])
        ax.set_ylim(ylim)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'results/aggregate_results/figures/class_split_{metric}.png', bbox_inches='tight')
    plt.show()

---

# Part 2: Isolating the Imbalance Effect (Control Analysis)

**Goal**: Prove that quality degradation is due to IMBALANCE, not just reduced dataset size.

## 2.1 Imbalanced vs. Size-Matched Control Comparison

In [ ]:
def compare_imbalanced_vs_control(imb_df, ctrl_df, metric='f1_minority'):
    """
    Compare imbalanced vs control conditions to isolate pure imbalance effect.
    """
    results = []
    
    for ir in IMBALANCE_RATIOS:
        for gen in GENERATORS:
            imb_vals = imb_df[(imb_df['ir_numeric'] == ir) & (imb_df['model'] == gen)][metric]
            ctrl_vals = ctrl_df[(ctrl_df['ir_numeric'] == ir) & (ctrl_df['model'] == gen)][metric]
            
            imb_mean = imb_vals.mean()
            ctrl_mean = ctrl_vals.mean()
            
            # Statistical test
            if len(imb_vals) > 0 and len(ctrl_vals) > 0:
                stat, p_value = stats.mannwhitneyu(imb_vals, ctrl_vals, alternative='two-sided')
            else:
                p_value = np.nan
            
            results.append({
                'IR': ir,
                'Generator': gen.upper(),
                'Imbalanced Mean': imb_mean,
                'Control Mean': ctrl_mean,
                'Difference (Imb - Ctrl)': imb_mean - ctrl_mean,
                'p-value': p_value,
                'Significant': p_value < 0.05 if not np.isnan(p_value) else False
            })
    
    return pd.DataFrame(results)

imb_vs_ctrl = compare_imbalanced_vs_control(imbalanced_results, control_results)
imb_vs_ctrl[imb_vs_ctrl['IR'].isin([1, 10, 50, 100])].round(4)

In [ ]:
def plot_imbalanced_vs_control_one(imb_df, ctrl_df, metric='f1_minority'):
    """
    Side-by-side comparison of imbalanced vs control conditions.
    """
    fig, axes = plt.subplots(figsize=(7, 5))

    def draw_panel(ax, imb_subset, ctrl_subset, title):
        imb_data = imb_subset.groupby('ir_numeric')[metric].agg(['mean', 'std']).reset_index()
        ctrl_data = ctrl_subset.groupby('ir_numeric')[metric].agg(['mean', 'std']).reset_index()

        ax.errorbar(imb_data['ir_numeric'], imb_data['mean'], yerr=imb_data['std'],
                    marker='o', linestyle='-', color='crimson', label='Imbalanced',
                    capsize=3, linewidth=2, markersize=8)
        ax.errorbar(ctrl_data['ir_numeric'], ctrl_data['mean'], yerr=ctrl_data['std'],
                    marker='s', linestyle='--', color='steelblue', label='Control (Size-matched)',
                    capsize=3, linewidth=2, markersize=8)

        ax.set_xlabel('Imbalance Ratio')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(title)
        ax.set_xscale('log')
        ax.set_xticks(IMBALANCE_RATIOS)
        ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS], rotation=45)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)

    # 6th panel: all generators
    draw_panel(axes, imb_df, ctrl_df, 'All generators')

    plt.suptitle('',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig
fig = plot_imbalanced_vs_control_one(imbalanced_results, control_results)
plt.savefig('results/aggregate_results/figures/fig9_imbalanced_vs_control_all.png', bbox_inches='tight')
plt.show()

---

# Part 3: Conditional Generation Compliance

During generation, we provided a conditional vector specifying a balanced class distribution. The conditional optimizes the output generation, but does not guarantee the samples are only from that condition. This analysis checks how well each generator adheres to the requested balanced distribution.

In [ ]:
processed_target_features = {
        "auto_mpg": "binaryClass",
        "credit": "target",
        "mammographic_mass": "Severity",
        "monks": "'class'",
        "phishing": "target",
        "vote": "target"
}   

orig_minority = {}
for dataset in result_files:
    f = Path(f'data/processed/{dataset}/train_imbalanced_ir_10_rep1.csv')
    if f.exists():
        df_ = pd.read_csv(f)
        orig_minority[dataset] = df_[processed_target_features[dataset]].value_counts().idxmin()
print(orig_minority)

def analyze_synthetic_class_distribution():
    synthetic_dirs = {}
    for dataset, run in RUN_IDS.items():
        synthetic_dirs[dataset] = f'data/synthetic/{dataset}/{run}'
    
    results = []
    
    for dataset_name, synth_dir in synthetic_dirs.items():
        synth_path = Path(synth_dir)
        if not synth_path.exists():
            continue
            
        for csv_file in synth_path.glob('*.csv'):
            filename = csv_file.name
            match = re.match(r'train_(\w+)_ir_(\d+)_rep(\d+)_synthetic_(\w+)\.csv', filename)
            if not match:
                continue
            
            dataset_type, ir, rep, model = match.groups()
            
            try:
                df = pd.read_csv(csv_file)
                target_col = processed_target_features[dataset_name]
                if target_col not in df.columns:
                    continue
                counts = df[target_col].value_counts()
                total = len(df)

                results.append({
                    'dataset': dataset_name,
                    'dataset_type': dataset_type,
                    'ir_numeric': int(ir),
                    'repetition': int(rep),
                    'model': model,
                    'total_samples': total,
                    'minority_share': counts.get(orig_minority.get(dataset_name), 0) / counts.sum(),
                })
            except Exception as e: print(f"{csv_file.name}: {e}")
    
    return pd.DataFrame(results)

synth_dist = analyze_synthetic_class_distribution()

synth_imb = synth_dist[synth_dist['dataset_type'] == 'imbalanced'].copy()
synth_imb = synth_imb[(synth_imb['minority_share'] > 0) & (synth_imb['minority_share'] < 1)]
r = (1 - synth_imb['minority_share']) / synth_imb['minority_share']
synth_imb['output_ir'] = np.maximum(r, 1/r)

In [ ]:
strip_df = synth_dist[synth_dist['dataset_type'] == 'imbalanced'].merge(
    imbalanced_results[['dataset', 'model', 'ir_numeric', 'repetition_id', 'f1_minority']]
        .rename(columns={'repetition_id': 'repetition'}),
    on=['dataset', 'model', 'ir_numeric', 'repetition'],
    how='left'
).dropna(subset=['f1_minority'])

fig, ax = plt.subplots(figsize=(12, 7))

sns.stripplot(data=strip_df, x='ir_numeric', y='minority_share',
              hue='f1_minority', palette='viridis',
              order=IMBALANCE_RATIOS,
              jitter=0.3, size=3.5, alpha=0.7, legend=False,
              ax=ax)

strip_df['ir_numeric']= strip_df['ir_numeric'].astype(dtype=object)

# colorbar
norm = plt.Normalize(0, 1)
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
cbar = fig.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label('F1 (Minority)')

ax.axhline(0.5, color='red', linestyle='--', linewidth=1.2, alpha=0.5)
ax.set_xticks(range(len(IMBALANCE_RATIOS)))
ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS])
ax.set_xlabel('Input Imbalance Ratio')
ax.set_ylabel('Generated Minority Class Share')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
ax.set_title('',
             fontsize=14, fontweight='bold')
plt.savefig('results/aggregate_results/figures/scatter_plot_f1_minority_ir.png', bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
def plot_per_dataset_profiles(df, control_df=None, metric='f1_minority'):
    """
    Line plot showing degradation profile for each dataset separately.
    """
    datasets = df['dataset'].unique()
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, dataset in enumerate(datasets):
        ax = axes[idx]
        ds_data = df[df['dataset'] == dataset]

        if control_df is not None:
            ctrl = control_df[control_df['dataset'] == dataset]
            ctrl_agg = ctrl.groupby('ir_numeric')[metric].mean().reset_index()
            if not ctrl_agg.empty:
                ax.plot(ctrl_agg['ir_numeric'], ctrl_agg[metric],
                        color='dimgray', linestyle='--', linewidth=2.5,
                        marker=None, markersize=7,
                        label='Control (balanced)', zorder=1, alpha=0.9)
        
        for gen in GENERATORS:
            gen_data = ds_data[ds_data['model'] == gen]
            agg = gen_data.groupby('ir_numeric')[metric].agg(['mean', 'std']).reset_index()

            agg['x'] = agg['ir_numeric']
            agg = agg.sort_values('ir_numeric')
            print(f"{dataset}/{gen}: {agg['x'].round(2).tolist()}")
            
            style = GENERATOR_STYLES[gen]
            ax.plot(agg['x'], agg['mean'],
                   marker=style['marker'], linestyle=style['linestyle'],
                   color=style['color'], label=style['label'],
                   linewidth=2, markersize=6)
            ax.fill_between(agg['x'],
                           agg['mean'] - agg['std'],
                           agg['mean'] + agg['std'],
                           alpha=0.15, color=style['color'])
        
        ax.set_xlabel('Imbalance Ratio')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(f'{dataset.replace("_", " ").title()}')
        ax.set_xscale('log')
        ax.set_xticks(IMBALANCE_RATIOS)
        ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS], rotation=45, fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)
        ax.set_xlim(0.8, 120)
    
    # Hide empty subplot
    if len(datasets) < len(axes):
        for i in range(len(datasets), len(axes)):
            axes[i].set_visible(False)
    
    # Single legend
    handles, labels = axes[0].get_legend_handles_labels()
    plt.suptitle('',
                fontsize=14, fontweight='bold', y=1.05)
    fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout()
    return fig

fig = plot_per_dataset_profiles(imbalanced_results, control_df=control_results)
plt.savefig('results/aggregate_results/figures/fig7_per_dataset_profiles.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for gen in GENERATORS:
    gen_data = synth_dist[synth_dist['model'] == gen]
    agg = gen_data.groupby('ir_numeric')['minority_share'].agg(['mean', 'std']).reset_index()
    style = GENERATOR_STYLES[gen]
    ax.plot(agg['ir_numeric'], agg['mean'],
            marker=style['marker'], linestyle=style['linestyle'],
            color=style['color'], label=style['label'],
            linewidth=2, markersize=8)
    ax.fill_between(agg['ir_numeric'],
                    agg['mean'] - agg['std'],
                    agg['mean'] + agg['std'],
                    alpha=0.15, color=style['color'])

ax.axhline(y=0.5, color='darkred', linestyle='--', linewidth=1.5, label='Target (50/50)', alpha=0.55)
ax.set_xlabel('Imbalance Ratio')
ax.set_ylabel('Generated Minority Class Share')
ax.set_title('Source versus Generated Class Distribution')
ax.set_xscale('log')
ax.set_xticks(IMBALANCE_RATIOS)
ax.set_xticklabels([f'{ir}:1' for ir in IMBALANCE_RATIOS], rotation=45)
ax.legend()
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/aggregate_results/figures/fig10_conditional_compliance.png', bbox_inches='tight')
plt.show()